# **Table of Contents**
* [Your Notebook](#your-notebook)
  * [Table of Contents](#table-of-contents)
* [TOC Generator](#toc-generator)


imports

In [1]:
import pandas as pd

## Clean Poverty Data

In [2]:
df_poverty = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_poverty_by_tract_s1701.csv')

df_poverty.columns

Index(['GEO_ID', 'NAME', 'S1701_C01_001E', 'S1701_C01_001M', 'S1701_C01_002E',
       'S1701_C01_002M', 'S1701_C01_003E', 'S1701_C01_003M', 'S1701_C01_004E',
       'S1701_C01_004M',
       ...
       'S1701_C03_058M', 'S1701_C03_059E', 'S1701_C03_059M', 'S1701_C03_060E',
       'S1701_C03_060M', 'S1701_C03_061E', 'S1701_C03_061M', 'S1701_C03_062E',
       'S1701_C03_062M', 'Unnamed: 374'],
      dtype='object', length=375)

In [3]:
df_poverty = df_poverty[["GEO_ID", "NAME", "S1701_C01_001E", "S1701_C02_001E", "S1701_C03_001E"]]

# The first row labels are redundant
df_poverty.drop(index = 0, inplace = True)

# Rename to useful column names
df_poverty = df_poverty.rename(columns={
    "NAME" : "TRACT",
    "S1701_C01_001E" : "POPULATION",
    "S1701_C02_001E" : "BELOW_POVERTY",
    "S1701_C03_001E" : "POVERTY_RATE"
    })

# Keep only tract # in string
df_poverty['TRACT'] = df_poverty['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Treat Tract # as a float for merging later
df_poverty['TRACT'] = df_poverty['TRACT'].astype(float)

df_poverty.head()

,GEO_ID,TRACT,POPULATION,BELOW_POVERTY,POVERTY_RATE
1,1400000US21111000201,2.01,1163,398,34.2
2,1400000US21111000202,2.02,706,370,52.4
3,1400000US21111000300,3.00,2155,585,27.1
4,1400000US21111000400,4.00,4644,955,20.6
5,1400000US21111000600,6.00,1252,328,26.2


In [4]:
df_poverty.to_csv("../Curtis/data/processed/poverty_rate_clean.csv", index = False)

## Clean Income Data

In [5]:
df_income = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_household_income_by_tract_b19013.csv')

df_income.head()

,Label (Grouping),Median household income in the past 12 months (in 2024 inflation-adjusted dollars)
0,Census Tract 2.01; Jefferson County; Kentucky,NaN
1,Estimate,"35,825"
2,Margin of Error,"±11,473"
3,Census Tract 2.02; Jefferson County; Kentucky,NaN
4,Estimate,"17,039"


In [6]:
# Rename columns
df_income = df_income.rename(columns={
    "Label (Grouping)" : "TRACT",
    "Median household income in the past 12 months (in 2024 inflation-adjusted dollars)" : "ESTIMATED_MEDIAN_HOUSEHOLD_INCOME"
    })

df_income.head()

,TRACT,ESTIMATED_MEDIAN_HOUSEHOLD_INCOME
0,Census Tract 2.01; Jefferson County; Kentucky,NaN
1,Estimate,"35,825"
2,Margin of Error,"±11,473"
3,Census Tract 2.02; Jefferson County; Kentucky,NaN
4,Estimate,"17,039"


In [7]:
# Shift values up one row so Estimate will be on the same row as tract
df_income['ESTIMATED_MEDIAN_HOUSEHOLD_INCOME'] = df_income['ESTIMATED_MEDIAN_HOUSEHOLD_INCOME'].shift(-1)

df_income.head()

,TRACT,ESTIMATED_MEDIAN_HOUSEHOLD_INCOME
0,Census Tract 2.01; Jefferson County; Kentucky,"35,825"
1,Estimate,"±11,473"
2,Margin of Error,NaN
3,Census Tract 2.02; Jefferson County; Kentucky,"17,039"
4,Estimate,"±6,250"


In [8]:
# Simplify the layout to be "Tract #: Estimate value" and delete Margin of Error rows
df_income = df_income.drop(df_income[df_income['TRACT'].str.contains('Estimate|Margin of Error', na=False)].index)

# Keep only tract # in string
df_income['TRACT'] = df_income['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Treat Tract # as a float for merging later
df_income['TRACT'] = df_income['TRACT'].astype(float)

# Make Tract the new index
df_income = df_income.set_index('TRACT')

df_income.head()

,ESTIMATED_MEDIAN_HOUSEHOLD_INCOME
TRACT,
2.01,"35,825"
2.02,"17,039"
3.00,"34,764"
4.00,"41,713"
6.00,"30,734"


In [9]:
df_income.to_csv("../Curtis/data/processed/income_clean.csv", index = True)

## Clean Housing Age Data

In [10]:
df_housing_age = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_housing_age_by_tract_b25034.csv')

df_housing_age.head()

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Margin of Error,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Margin of Error,Census Tract 6; Jefferson County; Kentucky!!Estimate,...,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Margin of Error,Census Tract 9801; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Margin of Error
0,Total:,777,±125,519,±118,"1,085",±149,"2,045",±230,866,...,"2,553",±264,"1,366",±225,"1,262",±252,503,±55,0,±13
1,Built 2020 or later,8,±14,0,±13,0,±13,0,±13,36,...,76,±81,0,±13,0,±13,2,±2,0,±13
2,Built 2010 to 2019,0,±13,0,±13,53,±79,0,±13,7,...,1,±10,64,±80,31,±39,4,±5,0,±13
3,Built 2000 to 2009,15,±22,14,±17,0,±13,42,±50,159,...,615,±239,0,±13,0,±13,5,±4,0,±13
4,Built 1990 to 1999,47,±69,39,±40,42,±20,0,±13,118,...,448,±241,17,±24,22,±35,4,±4,0,±13


In [11]:
# Drop all Estimate columns
cols_to_drop = df_housing_age.columns[df_housing_age.columns.str.contains('Estimate')]
df_housing_age = df_housing_age.drop(columns=cols_to_drop)

# Swap columns and rows to match other data sets
df_housing_age = df_housing_age.transpose()

# Set the first row as the header
df_housing_age = df_housing_age.rename(columns=df_housing_age.iloc[0]).drop(df_housing_age.index[0])

df_housing_age.head()

,Total:,Built 2020 or later,Built 2010 to 2019,Built 2000 to 2009,Built 1990 to 1999,Built 1980 to 1989,Built 1970 to 1979,Built 1960 to 1969,Built 1950 to 1959,Built 1940 to 1949,Built 1939 or earlier
Census Tract 2.01; Jefferson County; Kentucky!!Margin of Error,±125,±14,±13,±22,±69,±46,±54,±23,±61,±42,±132
Census Tract 2.02; Jefferson County; Kentucky!!Margin of Error,±118,±13,±13,±17,±40,±13,±13,±14,±12,±19,±117
Census Tract 3; Jefferson County; Kentucky!!Margin of Error,±149,±13,±79,±13,±20,±13,±63,±49,±63,±83,±131
Census Tract 4; Jefferson County; Kentucky!!Margin of Error,±230,±13,±13,±50,±13,±68,±155,±26,±206,±79,±259
Census Tract 6; Jefferson County; Kentucky!!Margin of Error,±125,±40,±14,±84,±76,±111,±14,±45,±37,±54,±76


In [12]:
# Remove the spaces before "Built" from all column names
df_housing_age.columns = df_housing_age.columns.str.strip()

# Remove the colon from "Total:"
df_housing_age.rename(columns={'Total:': 'TOTAL'}, inplace=True)

# Create index column
df_housing_age = df_housing_age.reset_index() 

# Now we can label first column
df_housing_age.rename(columns={df_housing_age.columns[0]: 'TRACT'}, inplace=True)

# Keep only tract # in string
df_housing_age['TRACT'] = df_housing_age['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Make Tract the new index
df_housing_age = df_housing_age.set_index('TRACT')

df_housing_age.head()

,TOTAL,Built 2020 or later,Built 2010 to 2019,Built 2000 to 2009,Built 1990 to 1999,Built 1980 to 1989,Built 1970 to 1979,Built 1960 to 1969,Built 1950 to 1959,Built 1940 to 1949,Built 1939 or earlier
TRACT,,,,,,,,,,,
2.01,±125,±14,±13,±22,±69,±46,±54,±23,±61,±42,±132
2.02,±118,±13,±13,±17,±40,±13,±13,±14,±12,±19,±117
3,±149,±13,±79,±13,±20,±13,±63,±49,±63,±83,±131
4,±230,±13,±13,±50,±13,±68,±155,±26,±206,±79,±259
6,±125,±40,±14,±84,±76,±111,±14,±45,±37,±54,±76


In [13]:
# Remove "+-" (non-digits \D) from the cell values in all columns
df_housing_age = df_housing_age.apply(lambda col: col.str.replace(r'[±]', '', regex = True))

# Convert values to float for use later
df_housing_age = df_housing_age.astype(int)

df_housing_age.head()

,TOTAL,Built 2020 or later,Built 2010 to 2019,Built 2000 to 2009,Built 1990 to 1999,Built 1980 to 1989,Built 1970 to 1979,Built 1960 to 1969,Built 1950 to 1959,Built 1940 to 1949,Built 1939 or earlier
TRACT,,,,,,,,,,,
2.01,125,14,13,22,69,46,54,23,61,42,132
2.02,118,13,13,17,40,13,13,14,12,19,117
3,149,13,79,13,20,13,63,49,63,83,131
4,230,13,13,50,13,68,155,26,206,79,259
6,125,40,14,84,76,111,14,45,37,54,76


In [14]:
df_housing_age.to_csv("../Curtis/data/processed/housing_age_clean.csv", index = True)

# TOC Generator 

Do not delete code. It will read your markdown and generate a TOC we can put in the notebook for easy navigation. 

In [15]:
import json
import os


def generate_toc_from_notebook(notebook_path):
    """
    Parses a local .ipynb file and generates Markdown for a Table of Contents.
    """
    if not os.path.isfile(notebook_path):
        print(f"❌ Error: File not found at '{notebook_path}'")
        return

    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)

    toc_markdown = "# **Table of Contents**\n"
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') == 'markdown':
            for line in cell.get('source', []):
                if line.strip().startswith('#'):
                    level = line.count('#')
                    title = line.strip('#').strip()
                    link = title.lower().replace(' ', '-').strip('-.()')
                    indent = '  ' * (level - 1)
                    toc_markdown += f"{indent}* [{title}](#{link})\n"

    print("\n--- ✅ Copy the Markdown below and paste"
          "it into a new markdown cell ---\n")
    print(toc_markdown)


if __name__ == "__main__":
    # Example usage
    notebook_path = 'curtis.ipynb'  # Replace with your notebook path
    generate_toc_from_notebook(notebook_path)


--- ✅ Copy the Markdown below and pasteit into a new markdown cell ---

# **Table of Contents**
* [**Table of Contents**](#**table-of-contents**)
      * [Clean Poverty Data](#clean-poverty-data)
      * [Clean Income Data](#clean-income-data)
* [TOC Generator](#toc-generator)

